# Actividad 7 - Variabilidad y casos extremos

### Importar librerías

In [ ]:
# Se importan las librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch

### Se cargan archivos de CSV de columnas relevantes a analizar

In [16]:

#Se cargan los archivos CSV relevantes
df_demographics = pd.read_csv("demographics.csv",encoding='utf-8-sig')
df_plantilla = pd.read_csv("plantilla_global.csv",encoding='utf-8-sig')
df_succession = pd.read_csv("succession.csv",encoding='utf-8-sig')

In [17]:
# Se convierten las columnas a tipo numérico, manejando errores de conversión
num_cols_plant = ["SDO PESOS MXN", "Sueldo USD", "BONOS Y COM ORI", "POSICI", "IPE"]
# Bucles para convertir las columnas numéricas de los DataFrames a tipo numérico, manejando errores de conversión
for col in num_cols_plant:
    df_plantilla[col] = pd.to_numeric(df_plantilla[col], errors="coerce")
df_succession["Edad"] = pd.to_numeric(df_succession["Edad"], errors="coerce")

In [18]:
# Variables de análisis con metadata
VARIABLES = {
    "Age": {
        "series"  : df_demographics["Age"],
        "label"   : "Edad del empleado",
        "unit"    : "años",
        "source"  : "Demographics",
        "cv_valid": True,
    },
    "Tenure_Alfa": {
        "series"  : df_demographics["Tenure (Years) Alfa"],
        "label"   : "Antigüedad en Alfa",
        "unit"    : "años",
        "source"  : "Demographics",
        "cv_valid": True,
    },
    "Tenure_Sigma": {
        "series"  : df_demographics["Tenure (Years) Sigma"],
        "label"   : "Antigüedad en Sigma",
        "unit"    : "años",
        "source"  : "Demographics",
        "cv_valid": True,
    },
    "Sueldo_MXN": {
        "series"  : df_plantilla["SDO PESOS MXN"],
        "label"   : "Sueldo mensual (MXN)",
        "unit"    : "MXN",
        "source"  : "Plantilla Global",
        "cv_valid": True,
    },
    "Bonos_MXN": {
        "series"  : df_plantilla["BONOS Y COM ORI"],
        "label"   : "Bonos y comisiones (MXN)",
        "unit"    : "MXN",
        "source"  : "Plantilla Global",
        "cv_valid": True,
    },
    "Comp_Ratio": {
        "series"  : df_plantilla["POSICI"],
        "label"   : "Compa Ratio / Posicionamiento",
        "unit"    : "%",
        "source"  : "Plantilla Global",
        "cv_valid": True,
    },
    "IPE": {
        "series"  : df_plantilla["IPE"],
        "label"   : "IPE (valuación de puesto)",
        "unit"    : "puntos",
        "source"  : "Plantilla Global",
        "cv_valid": True,
    },
    "Edad_Incumbent": {
        "series"  : df_succession["Edad"],
        "label"   : "Edad del titular (Succession)",
        "unit"    : "años",
        "source"  : "Succession",
        "cv_valid": True,
    },
}

In [19]:
# Validación de datos cargados
print(f"\nDatos cargados: {len(df_demographics)} empleados · {len(df_succession)} nominaciones")
print(f"   Variables a analizar: {len(VARIABLES)}\n")


Datos cargados: 50000 empleados · 38200 nominaciones
   Variables a analizar: 8



In [20]:
def analisis_dispersion(series: pd.Series, name: str, meta: dict) -> dict:
    """
    Calcula medidas de dispersión para una serie de datos númericos

    Retorna diccionario con:
    a) Medidas de tendencia central: media, mediana, moda
    b) Medidas de dispersión: std, varianza, rango, IQR, CV
    c) Cuartiles: Q1, Q2, Q3
    d) Coeficiente de variación (CV) - solo si la media es mayor de cero y sin valores negativos
    e) Cercas IQR: lower_fence, upper_fence
    f) Valores atípicos: outliers
    """

    # Elimina valores nulos para el análisis
    s= series.dropna().astype(float)  
    
    # Se obtiene el tamaño de la muestra
    n = len(s)

    """ a) Medidas de tendencia central """
    # Media
    mean = s.mean()

    # Mediana
    median = s.median()

    # Moda
    # Se obtiene la moda, si existe; de lo contrario, se asigna NaN
    mode = s.mode().iloc[0] if not s.mode().empty else np.nan

    """ b) Medidas de dispersión """

    # Desviación estándar (muestra)
    std = s.std(ddof=1)

    #Varianza
    variance = s.var(ddof=1)

    # Rango
    rng = s.max() - s.min()

    """ c) Cuartiles """
    # Q1 (Cuarto 1)
    q1 = s.quantile(0.25)

    # Q2 (Cuarto 2) igual a la mediana
    q2 = s.quantile(0.50)

    # Q3 (Cuarto 3)
    q3 = s.quantile(0.75)

    # Rango intercuartílico
    iqr = q3 - q1  

    # Percentil 10
    p10 = s.quantile(0.10)

    # Percentil 90
    p90 = s.quantile(0.90)

    """ d) Coeficiente de variación (CV) """
    # Se evalua si la media es mayor que cero y no hay valores negativos en la serie
    # En caso de ser verdadero, se calcula el CV; de lo contrario, se asigna None
    cv = std / mean * 100 if (meta["cv_valid"] and mean > 0) else None

    """e) Cercas IQR """
    # Cerca inferior
    lower_fence = q1 - 1.5 * iqr

    # Cerca superior
    upper_fence = q3 + 1.5 * iqr

    #Cerca inferior extrema
    lower_fence_extreme = q1 - 3.0 * iqr
    
    # Cerca superior extrema
    upper_fence_extreme = q3 + 3.0 * iqr

    """f) Valores atípicos """
    # Valores atípicos por debajo de la cerca inferior
    outliers_low = s[s < lower_fence]

    # Valores atípicos por encima de la cerca superior
    outliers_high = s[s > upper_fence]

    # Valores atípicos por encima de la cerca superior extrema
    extreme_low = s[s < lower_fence_extreme]

    # Valores atípicos por debajo de la cerca inferior extrema
    extreme_high = s[s > upper_fence_extreme]

    # Número total de valores atípicos
    n_out = len(outliers_low) + len(outliers_high)

    # Número total de valores atípicos extremos
    n_ext = len(extreme_low) + len(extreme_high)

    # Porcentaje de valores atípicos
    pct_out = (n_out / n * 100) if n > 0 else 0

    # Cálculo de la asimetría (skewness)
    skew = s.skew()

    # Cálculo de la curtosis (kurtosis)
    kurt = s.kurtosis()

    # Se retorna un diccionario con todas las medidas calculadas
    return {
        "name": name,
        "label": meta["label"],
        "unit": meta["unit"],
        "source": meta["source"],
        "n": n,
        "mean": mean,
        "median": median,
        "mode": mode,
        "std": std,
        "variance": variance,
        "range": rng,
        "q1": q1,
        "q2": q2,
        "q3": q3,
        "iqr": iqr,
        "p10": p10,
        "p90": p90,
        "cv": cv,
        "lower_fence": lower_fence,
        "upper_fence": upper_fence,
        "lower_fence_extreme": lower_fence_extreme,
        "upper_fence_extreme": upper_fence_extreme,
        "n_outliers_low": len(outliers_low),
        "n_outliers_high": len(outliers_high),
        "n_outliers_total": n_out,
        "n_extreme_total": n_ext,
        "pct_outliers": pct_out,
        "outliers_low_vals": sorted(outliers_low.tolist()),
        "outliers_high_vals": sorted(outliers_high.tolist(), reverse=True),
        "skew": skew,
        "kurtosis": kurt,
        "series": s,
    }


In [21]:
# Se realiza el bucle para analizar cada variable definida en el diccionario VARIABLES 
# y almacenar los resultados en un nuevo diccionario results
results = {}

for var_name, meta in VARIABLES.items():
    results[var_name] = analisis_dispersion(meta["series"], var_name, meta)

In [22]:
def fmt(val, unit, decimals=2):
    """
    Formatea un valor numérico según su unidad de medida.
    """
    # Si el valor es None, se retorna "N/A"
    if val is None:
        return "N/A"
    # Si la unidad es "MXN", se formatea como moneda mexicana
    if unit == "MXN":
        return f"${val:,.0f}"
    # Si la unidad es "%", se formatea como porcentaje
    if unit == "%":
        return f"{val:.{decimals}f}%"
    # Por defecto, se formatea con el número de decimales especificado
    return f"{val:.{decimals}f} {unit}".strip()

In [23]:
# Medidas de dispersión para cada variable
for name, r in results.items():
    # Nombre y fuente de la variable
    print(f"\n {r['label']} ({r['source']})")
    # Se imprimen las medidas de dispersión calculadas
    print(f"   N: {r['n']}")
    print(f"   Media: {fmt(r['mean'], r['unit'])}")
    print(f"   Mediana: {fmt(r['median'], r['unit'])}")
    print(f"   Moda: {fmt(r['mode'], r['unit'])}")
    print(f"   Desviación estándar: {fmt(r['std'], r['unit'])}")
    print(f"   Varianza: {fmt(r['variance'], r['unit'])}")
    print(f"   Rango: {fmt(r['range'], r['unit'])}")
    print(f"   Q1: {fmt(r['q1'], r['unit'])}")
    print(f"   Q2 (Mediana): {fmt(r['q2'], r['unit'])}")
    print(f"   Q3: {fmt(r['q3'], r['unit'])}")
    print(f"   IQR: {fmt(r['iqr'], r['unit'])}")
    print(f"   Percentil 10: {fmt(r['p10'], r['unit'])}")
    print(f"   Percentil 90: {fmt(r['p90'], r['unit'])}")
    # Coeficiente de variacion
    if r["cv"] is not None:
        print(f"   Coeficiente de variación (CV): {fmt(r['cv'], '%')}")
    else:
        print("   Coeficiente de variación (CV): N/A")
    print(f"   Cercas IQR: [{fmt(r['lower_fence'], r['unit'])}, {fmt(r['upper_fence'], r['unit'])}]")
    print(f"   Valores atípicos por debajo de la cerca inferior: {r['n_outliers_low']}")
    print(f"   Valores atípicos por encima de la cerca superior: {r['n_outliers_high']}")
    print(f"   Total de valores atípicos: {r['n_outliers_total']} ({r['pct_outliers']:.2f}%)")

    # Asimetría y curtosis
    print(f"   Asimetría (Skewness): {r['skew']:.2f}")
    print(f"   Curtosis (Kurtosis): {r['kurtosis']:.2f}")


 Edad del empleado (Demographics)
   N: 50000
   Media: 40.42 años
   Mediana: 40.00 años
   Moda: 26.00 años
   Desviación estándar: 9.54 años
   Varianza: 91.02 años
   Rango: 33.00 años
   Q1: 32.00 años
   Q2 (Mediana): 40.00 años
   Q3: 49.00 años
   IQR: 17.00 años
   Percentil 10: 27.00 años
   Percentil 90: 54.00 años
   Coeficiente de variación (CV): 23.60%
   Cercas IQR: [6.50 años, 74.50 años]
   Valores atípicos por debajo de la cerca inferior: 0
   Valores atípicos por encima de la cerca superior: 0
   Total de valores atípicos: 0 (0.00%)
   Asimetría (Skewness): 0.01
   Curtosis (Kurtosis): -1.20

 Antigüedad en Alfa (Demographics)
   N: 50000
   Media: 13.70 años
   Mediana: 13.70 años
   Moda: 21.50 años
   Desviación estándar: 6.75 años
   Varianza: 45.60 años
   Rango: 23.40 años
   Q1: 7.80 años
   Q2 (Mediana): 13.70 años
   Q3: 19.50 años
   IQR: 11.70 años
   Percentil 10: 4.40 años
   Percentil 90: 23.00 años
   Coeficiente de variación (CV): 49.28%
   Cercas IQ

In [24]:
# Clasificación del coeficiente de variación (CV) en categorías
def cv_class(cv):
    """
    Clasifica el coeficiente de variación (CV) en categorías según su valor.
    """
    if cv is None:
        return "N/A"
    if cv < 20:
        return "Muy baja (<20%)"
    elif cv < 35:
        return "Baja (20–35%)"
    elif cv < 50:
        return "Media (35–50%)"
    else:
        return "Alta (>50%)"
    
# Se crea una lista de tuplas con el nombre de la variable, su CV y su clasificación
cv_data = [(name, r["cv"], r["label"]) for name, r in results.items() if r["cv"] is not None]
# Se acomoda la lista de tuplas por el valor del CV en orden descendente
cv_data.sort(key=lambda x: x[1] if x[1] else 0, reverse=True)

# Se imprime la tabla de CV y su clasificación
print(f"\n  {'Variable':<20} {'CV':>8}   Clasificación")
# Se imprime una línea divisoria para la tabla
print(f"  {'─'*20} {'─'*8}   {'─'*25}")
# Se imprime cada variable con su CV y clasificación
for name, cv, label in cv_data:
    print(f"  {name:<20} {cv:>7.1f}%   {cv_class(cv)}")


  Variable                   CV   Clasificación
  ──────────────────── ────────   ─────────────────────────
  Bonos_MXN               94.7%   Alta (>50%)
  Sueldo_MXN              89.9%   Alta (>50%)
  Tenure_Sigma            69.3%   Alta (>50%)
  Tenure_Alfa             49.3%   Media (35–50%)
  IPE                     27.1%   Baja (20–35%)
  Age                     23.6%   Baja (20–35%)
  Edad_Incumbent          23.5%   Baja (20–35%)
  Comp_Ratio              15.9%   Muy baja (<20%)


In [25]:

# Se imprime la tabla de cuartiles y rango intercuartílico (IQR) para cada variable
print(f"\n  {'Variable':<18} {'Q1':>10} {'Q2(Med)':>10} {'Q3':>10} {'IQR':>10}")
# Se imprime una línea divisoria para la tabla
print(f"  {'─'*18} {'─'*10} {'─'*10} {'─'*10} {'─'*10}")
# Se imprime cada variable con sus cuartiles y rango intercuartílico
for name, r in results.items():
    u = r["unit"]
    print(f"  {name:<18} {fmt(r['q1'],u):>10} {fmt(r['q2'],u):>10} {fmt(r['q3'],u):>10} {fmt(r['iqr'],u):>10}")


  Variable                   Q1    Q2(Med)         Q3        IQR
  ────────────────── ────────── ────────── ────────── ──────────
  Age                32.00 años 40.00 años 49.00 años 17.00 años
  Tenure_Alfa         7.80 años 13.70 años 19.50 años 11.70 años
  Tenure_Sigma        3.20 años  6.10 años 10.80 años  7.60 años
  Sueldo_MXN           $101,926   $621,433 $1,362,060 $1,260,133
  Bonos_MXN                $665     $3,250     $7,186     $6,521
  Comp_Ratio             88.00%     99.70%    111.00%     23.00%
  IPE                10.00 puntos 12.00 puntos 16.00 puntos 6.00 puntos
  Edad_Incumbent     32.00 años 41.00 años 49.00 años 17.00 años


In [26]:
# Despliegue de los resultados de valores atípicos para cada variable
print(f"\n  {'Variable':<18} {'Cerca inf':>12} {'Cerca sup':>12} {'N atíp.':>8} {'%':>6}")
# Se imprime una línea divisoria para la tabla
print(f"  {'─'*18} {'─'*12} {'─'*12} {'─'*8} {'─'*6}")
# Bucle para imprimir cada variable con sus cercas IQR, número de valores atípicos y porcentaje
for name, r in results.items():
    u = r["unit"]
    flag = " ⚠️" if r["n_outliers_total"] > 0 else ""
    print(f"  {name:<18} {fmt(r['lower_fence'],u):>12} {fmt(r['upper_fence'],u):>12} "
          f"{r['n_outliers_total']:>8}{flag:2} {r['pct_outliers']:>5.1f}%")

# Se imprime el total de valores atípicos detectados en todas las variables
print(f"\n  Total de atípicos detectados: "
      f"{sum(r['n_outliers_total'] for r in results.values())}")


  Variable              Cerca inf    Cerca sup  N atíp.      %
  ────────────────── ──────────── ──────────── ──────── ──────
  Age                   6.50 años   74.50 años        0     0.0%
  Tenure_Alfa          -9.75 años   37.05 años        0     0.0%
  Tenure_Sigma         -8.20 años   22.20 años      444 ⚠️   0.9%
  Sueldo_MXN          $-1,788,274   $3,252,259        0     0.0%
  Bonos_MXN               $-9,117      $16,968       85 ⚠️   0.2%
  Comp_Ratio               53.50%      145.50%        0     0.0%
  IPE                 1.00 puntos 25.00 puntos        0     0.0%
  Edad_Incumbent        6.50 años   74.50 años        0     0.0%

  Total de atípicos detectados: 529


In [27]:
# Leyenda para la clasificación de los valores atípicos detectados en las variables analizadas
OUTLIER_CLASSIFICATION = {
    "Tenure_Alfa": {
        "tipo"    : "CASO EXTREMO LEGÍTIMO",
        "razon"   : "Empleados fundadores o transferidos desde fusión original. "
                    "Llevan 22–25 años en Alfa. No es un error de carga; "
                    "es patrimonio histórico de la empresa.",
        "accion"  : "Etiquetar como 'Empleado Histórico' en DIM_EMPLOYEE. "
                    "Usar mediana en reportes de retención.",
    },
    "Tenure_Sigma": {
        "tipo"    : "CASO EXTREMO LEGÍTIMO",
        "razon"   : "Empleados fundadores o transferidos desde fusión original. "
                    "Llevan 22–25 años en Sigma. No es un error de carga; "
                    "es patrimonio histórico de la empresa.",
        "accion"  : "Etiquetar como 'Empleado Histórico' en DIM_EMPLOYEE. "
                    "Usar mediana en reportes de retención.",
    },
    "Sueldo_MXN": {
        "tipo"    : "CASO EXTREMO LEGÍTIMO",
        "razon"   : "Empleados con sueldos > $200,000 MXN son casos de "
                    "sobrecompensación por antigüedad o desempeño. "
                    "No es un error de carga; es una situación real.",
        "accion"  : "Usar mediana en reportes de compensación. "
                    "Crear flag 'Sobrecompensado' en modelo de compensación.",
    },
    "Bonos_MXN": {
        "tipo"    : "CASO EXTREMO LEGÍTIMO",
        "razon"   : "Personal de ventas con esquemas de comisión variable o "
                    "directivos con bonos de desempeño excepcional. "
                    "Bonos entre $25,624 y $29,380 MXN son posibles en N1–N2.",
        "accion"  : "Segmentar por SUBTIPO/GRUPO antes de calcular promedios. "
                    "Crear flag 'Esquema_Variable' en modelo de compensación.",
    },
    "Comp_Ratio": {
        "tipo"    : "CASO EXTREMO LEGÍTIMO",
        "razon"   : "Empleados con Compa Ratio > 200% son casos de "
                    "sobrecompensación por antigüedad o desempeño. "
                    "No es un error de carga; es una situación real.",
        "accion"  : "Usar mediana en reportes de compensación. "
                    "Crear flag 'Sobrecompensado' en modelo de compensación.",
    },
    "IPE": {
        "tipo"    : "CASO EXTREMO LEGÍTIMO",
        "razon"   : "Empleados con IPE > 100 puntos son casos de "
                    "sobrecompensación por antigüedad o desempeño. "
                    "No es un error de carga; es una situación real.",
        "accion"  : "Usar mediana en reportes de compensación. "
                    "Crear flag 'Sobrecompensado' en modelo de compensación.",
    },
    "Edad_Incumbent": {
        "tipo"    : "CASO EXTREMO LEGÍTIMO",
        "razon"   : "Titulares con edad > 65 años son casos de "
                    "empleados con larga trayectoria o directivos. "
                    "No es un error de carga; es una situación real.",
        "accion"  : "Usar mediana en reportes de sucesión. "
                    "Crear flag 'Empleado Senior' en modelo de sucesión.",
    },
}

for var_name, r in results.items():
    # Si el numero total de valores atípicos es mayor a cero, 
    # se imprime un mensaje de advertencia con la clasificación correspondiente
    if r["n_outliers_total"] > 0:
        clf = OUTLIER_CLASSIFICATION.get(var_name, {
            "tipo"  : "PENDIENTE DE REVISIÓN",
            "razon" : "Requiere validación con el equipo de datos.",
            "accion": "Revisar con SAP/fuente original.",
        })
        # Se imprime la información de la variable con valores atípicos y su clasificación
        print(f"\n {r['label']}")
        print(f"     Atípicos : {r['n_outliers_total']} ({r['pct_outliers']:.1f}%)")
        print(f"     Valores  : {[round(v,2) for v in r['outliers_high_vals'][:5]]}")
        print(f"     Tipo     : {clf['tipo']}")
        print(f"     Razón    : {clf['razon']}")
        print(f"     Acción   : {clf['accion']}")

# Se identifican las variables sin valores atípicos
no_outliers = [n for n, r in results.items() if r["n_outliers_total"] == 0]
# Se imprime la lista de variables sin valores atípicos
print(f"\n Variables sin atípicos ({len(no_outliers)}):")
# Se imprime cada variable sin valores atípicos
for n in no_outliers:
    print(f"     • {results[n]['label']}")



 Antigüedad en Sigma
     Atípicos : 444 (0.9%)
     Valores  : [25.1, 25.1, 25.1, 25.0, 25.0]
     Tipo     : CASO EXTREMO LEGÍTIMO
     Razón    : Empleados fundadores o transferidos desde fusión original. Llevan 22–25 años en Sigma. No es un error de carga; es patrimonio histórico de la empresa.
     Acción   : Etiquetar como 'Empleado Histórico' en DIM_EMPLOYEE. Usar mediana en reportes de retención.

 Bonos y comisiones (MXN)
     Atípicos : 85 (0.2%)
     Valores  : [17961.97, 17866.67, 17830.61, 17816.3, 17786.12]
     Tipo     : CASO EXTREMO LEGÍTIMO
     Razón    : Personal de ventas con esquemas de comisión variable o directivos con bonos de desempeño excepcional. Bonos entre $25,624 y $29,380 MXN son posibles en N1–N2.
     Acción   : Segmentar por SUBTIPO/GRUPO antes de calcular promedios. Crear flag 'Esquema_Variable' en modelo de compensación.

 Variables sin atípicos (6):
     • Edad del empleado
     • Antigüedad en Alfa
     • Sueldo mensual (MXN)
     • Compa Ratio /

In [28]:
# Campos con mayor variabilidad (CV) e implicaciones para el análisis
# Se define un diccionario con las implicaciones de cada variable según su CV
IMPLICATIONS = {
    "Tenure_Sigma": (
        "ALTA",
        "La dispersión refleja perfiles muy heterogéneos de lealtad "
        "a Sigma. No usar la media como único representante — siempre "
        "presentarla junto con la mediana. Segmentar por Tenure_Band."
    ),
    "Bonos_MXN": (
        "ALTA",
        "Componente salarial más volátil del modelo. Los atípicos "
        "superiores son legítimos (ventas/directivos). Para benchmarks "
        "salariales, excluir o segmentar por esquema de compensación."
    ),
    "Tenure_Alfa": (
        "MEDIA",
        "Dispersión esperada en empresa con décadas de historia. "
        "Plantilla heterogénea: desde nuevos ingresos hasta veteranos "
        "de 25 años. Contexto válido para análisis de retención."
    ),
    "Sueldo_MXN": (
        "ALTA",
        "Alta en términos absolutos pero esperada dada la amplitud "
        "de niveles (N1–N6). El análisis de sueldos solo es significativo "
        "filtrado por SEGMENTO_P o NIVEL."
    ),
    "Comp_Ratio": (
        "MUY BAJA",
        "Variable más controlada y confiable del modelo. Ideal como "
        "KPI de equidad salarial en el dashboard ejecutivo. Neutraliza "
        "diferencias de escala entre regiones y monedas."
    ),

}

# Se ordenan las variables por su coeficiente de variación (CV) en orden descendente
variab_sorted = sorted(
    [(n, r["cv"]) for n, r in results.items() if r["cv"] is not None],
    key=lambda x: x[1], reverse=True
)

# Se crea bucle para imprimir cada variable con su CV, 
# clasificación y descripción de implicaciones
for name, cv in variab_sorted:
    r = results[name]
    imp = IMPLICATIONS.get(name)
    nivel = imp[0] if imp else cv_class(cv).split()[1]
    desc  = imp[1] if imp else "Sin implicación documentada."
    print(f"\n  {r['label']} · CV = {cv:.1f}% [{nivel}]")
    print(f"  → {desc}")


  Bonos y comisiones (MXN) · CV = 94.7% [ALTA]
  → Componente salarial más volátil del modelo. Los atípicos superiores son legítimos (ventas/directivos). Para benchmarks salariales, excluir o segmentar por esquema de compensación.

  Sueldo mensual (MXN) · CV = 89.9% [ALTA]
  → Alta en términos absolutos pero esperada dada la amplitud de niveles (N1–N6). El análisis de sueldos solo es significativo filtrado por SEGMENTO_P o NIVEL.

  Antigüedad en Sigma · CV = 69.3% [ALTA]
  → La dispersión refleja perfiles muy heterogéneos de lealtad a Sigma. No usar la media como único representante — siempre presentarla junto con la mediana. Segmentar por Tenure_Band.

  Antigüedad en Alfa · CV = 49.3% [MEDIA]
  → Dispersión esperada en empresa con décadas de historia. Plantilla heterogénea: desde nuevos ingresos hasta veteranos de 25 años. Contexto válido para análisis de retención.

  IPE (valuación de puesto) · CV = 27.1% [(20–35%)]
  → Sin implicación documentada.

  Edad del empleado · CV = 23

ipe, monthly_salary_mxn, monthly_salary_usd, bonuses_original, comp_ratio (df_plantilla)
age, antiguedad alfa, antiguedad sigma (demographics)
incumbent_age (df_succession)
score_overall, score_leadership, leadership_vision, leadership_influence (candidate)